### Importy

In [25]:
import torch
import onnxruntime as ort
import cv2
import numpy as np
import os
import pickle
from insightface.app import FaceAnalysis
from pathlib import Path

### Sprawdzanie GPU

In [11]:
cuda_available = torch.cuda.is_available()
print(f"PyTorch CUDA: {cuda_available}")
if cuda_available:
    print(f"Urządzenie: {torch.cuda.get_device_name(0)}")
    print(f"Liczba GPU: {torch.cuda.device_count()}")

PyTorch CUDA: True
Urządzenie: NVIDIA GeForce RTX 3060 Laptop GPU
Liczba GPU: 1


In [12]:
providers = ort.get_available_providers()
print(f"Dostępne procesory ONNX: {providers}")

if 'CUDAExecutionProvider' in providers:
    print("InsightFace będzie korzystać z GPU (CUDA).")
else:
    print("Brak CUDAExecutionProvider (CPU).")


Dostępne procesory ONNX: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
InsightFace będzie korzystać z GPU (CUDA).


### Inicjalizacja modelu

In [13]:
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\kubte/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'

### Funkcja do detekcji, alignmentu i ekstrakcji cech

In [50]:
def get_face_embedding(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    faces = app.get(img)
    
    if len(faces) == 0:
        print(f"Undetected face in the image: {image_path}")
        return None

    # Zwracamy embedding pierwszej wykrytej twarzy (zakładamy 1 osobę na foto)
    # faces[0].normed_embedding to wektor 512-wymiarowy
    return faces[0].normed_embedding

In [ ]:
emb1 = get_face_embedding('data/000001.jpg')
emb2 = get_face_embedding('data/000002.jpg')

In [24]:
emb1.shape

(512,)

### Wyliczenie miary podobieństwa (Cosinus similarity)

In [18]:
if emb1 is not None and emb2 is not None:
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    print(f"Podobieństwo: {similarity:.4f}")

Podobieństwo: -0.0046


### Sprawdzenie datasetu

In [22]:
from pathlib import Path

gallery_folder = Path(r"data_processed/gallery")
test_known_folder = Path(r"data_processed/test_known")
test_unknown_folder = Path(r"data_processed/test_unknown")

num_files_in_gallery = sum(1 for p in gallery_folder.iterdir() if p.is_file())
num_files_in_test_known = sum(1 for p in test_known_folder.iterdir() if p.is_file())
num_files_in_test_unknown = sum(1 for p in test_unknown_folder.iterdir() if p.is_file())

print("Files in gallery:", num_files_in_gallery)
print("Files in test_unknown:", num_files_in_test_unknown)
print("Files in test_known:", num_files_in_test_known)

Files in gallery: 102
Files in test_unknown: 120
Files in test_known: 1020


### Utworzenie wektorów osadzeń na podstawie zdjęć 102 użytkowników

In [63]:
aaron_eckhart_emb = get_face_embedding('data_processed/test_known/Alicja_9.jpeg')
aaron_eckhart_emb

Undetected face in the image: data_processed/test_known/Alicja_9.jpeg


In [70]:
def create_biometric_database(source_folder, output_file="face_db.pkl"):
    database = {}
        
    print("Processing...")
    
    for file_name in os.listdir(source_folder):
        if not file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
            
        person_id = os.path.splitext(file_name)[0]
        img_path = os.path.join(source_folder, file_name)
        
        img = cv2.imread(img_path)

        if img is None:
            print(f"Error: Could not read image {file_name}. Skipping.")
            continue

        h, w = img.shape[:2]
        pad_h, pad_w = int(h * 0.4), int(w * 0.4)
        img_padded = cv2.copyMakeBorder(
            img, pad_h, pad_h, pad_w, pad_w, 
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )

        faces = app.get(img_padded)
        
        if len(faces) > 0:
            # Sortujemy twarze po wielkości (na wypadek gdyby w tle był ktoś inny)
            # i bierzemy największą (główną) twarz
            faces = sorted(faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]), reverse=True)
            
            embedding = faces[0].normed_embedding
            
            database[person_id] = embedding
            print(f"Saved: {person_id}")
        else:
            print(f"Error saving: {person_id}")

    with open(output_file, "wb") as f:
        pickle.dump(database, f)
        
    print(f"\nDone! Embeddings saved in {output_file}")
    return database

In [73]:
create_biometric_database('data_processed/gallery/')

Processing...
Saved: Aaron_Eckhart_ref
Saved: Adrien_Brody_ref
Saved: Alexander_Skarsgard_ref
Saved: Alicja_ref
Saved: America_Ferrera_ref
Saved: Andrea_Bowen_ref
Saved: Angell_Conwell_ref
Saved: Angie_Harmon_ref
Saved: Antonio_Banderas_ref
Saved: Bernie_Mac_ref
Saved: Brendan_Fraser_ref
Saved: Christel_Khalil_ref
Saved: Ciara_Bravo_ref
Saved: Clint_Eastwood_ref
Saved: Colin_Farrell_ref
Saved: Crystal_Bernard_ref
Saved: Daniel_Radcliffe_ref
Saved: Danny_Trejo_ref
Saved: Dan_Lauria_ref
Saved: Dianna_Agron_ref
Saved: Edie_Falco_ref
Saved: Ed_Harris_ref
Saved: Elizabeth_Berkley_ref
Saved: Elizabeth_Hendrickson_ref
Saved: Eliza_Dushku_ref
Saved: Ellen_Greene_ref
Saved: Emile_Hirsch_ref
Saved: Farah_Fath_ref
Saved: Florencia_Lozano_ref
Saved: Gabrielle_Carteris_ref
Saved: Heather_Locklear_ref
Saved: J.K._Simmons_ref
Saved: Jakub_ref
Saved: James_Brolin_ref
Saved: James_Frain_ref
Saved: Jamie_Foxx_ref
Saved: Jamie_Luner_ref
Saved: Jane_Lynch_ref
Saved: Jared_Padalecki_ref
Saved: Jason_Batema

{'Aaron_Eckhart_ref': array([ 4.46673594e-02,  9.97412764e-03,  1.41310627e-02,  4.37175743e-02,
        -1.04909139e-02, -2.05486938e-02, -5.70709594e-02, -5.30050136e-02,
        -4.12916671e-03,  1.12755541e-02,  7.99206831e-03, -1.32724661e-02,
         5.24081476e-02, -5.84124215e-02,  4.18157130e-02, -4.39009280e-04,
        -5.75906634e-02, -3.34601998e-02,  1.51671488e-02, -5.62507734e-02,
         4.17294167e-02, -1.35154435e-02,  5.78899831e-02, -5.27283410e-03,
         2.18846761e-02,  5.17101921e-02, -7.84392282e-03,  4.04822901e-02,
        -1.85999162e-02,  6.28017709e-02, -1.86661165e-02,  4.86015640e-02,
        -3.87004539e-02,  1.12600848e-02,  4.66400757e-03,  3.94917317e-02,
        -9.32850596e-03, -3.88543867e-02, -1.92149654e-02,  3.59252356e-02,
        -1.93418451e-02,  1.09345727e-01, -1.06361052e-02,  2.63858829e-02,
         3.37890498e-02,  2.14494765e-02, -7.29343668e-02, -5.11205159e-02,
         7.79906986e-03, -7.26377368e-02,  4.68526967e-02, -2.57452

### Wczytanie bazy danych

In [74]:
with open("face_db.pkl", "rb") as f:
    db = pickle.load(f)

klucz = list(db.keys())[0]
wektor = db[klucz]

print(f"Osoba: {klucz}")
print(f"Typ danych: {wektor.dtype}")
print(f"Kształt wektora: {wektor.shape}")
print(f"Pierwsze 5 liczb wektora: {wektor[:5]}")

Osoba: Aaron_Eckhart_ref
Typ danych: float32
Kształt wektora: (512,)
Pierwsze 5 liczb wektora: [ 0.04466736  0.00997413  0.01413106  0.04371757 -0.01049091]
